# Ejercicios Tema 4: visualización de datos

Soluciones explicadas con Plotly Express y Graph Objects.

## Preparación de los datos

Como el enunciado no incluye un archivo de datos, se crea un conjunto de demostración reproducible con las columnas `edad`, `ingresos`, `region` y `genero`. Puedes sustituir esta celda por la carga de tu propio DataFrame, manteniendo esos nombres de columna. Los datos generados solo sirven para practicar y no representan una población real.

In [2]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

generador = np.random.default_rng(42)
n = 300

regiones = np.array(["Norte", "Centro", "Sur", "Este", "Oeste"])
regiones_fila = generador.choice(regiones, size=n)
generos = generador.choice(["Mujer", "Hombre"], size=n)
edades = np.clip(np.rint(generador.normal(loc=40, scale=12, size=n)), 18, 80).astype(int)

bonificacion_region = {"Norte": 4000, "Centro": 2500, "Sur": -2500, "Este": 1000, "Oeste": 0}
ingresos = (
    18000
    + edades * 550
    + np.array([bonificacion_region[region] for region in regiones_fila])
    + generador.normal(loc=0, scale=8500, size=n)
)
ingresos = np.clip(ingresos, 12000, None).round(2)

df = pd.DataFrame(
    {
        "edad": edades,
        "ingresos": ingresos,
        "region": regiones_fila,
        "genero": generos,
    }
)

display(df.head())
print("Dimensiones del DataFrame:", df.shape)

,edad,ingresos,region,genero
0,36,43801.31,Norte,Mujer
1,39,54144.49,Este,Hombre
2,37,28959.82,Este,Mujer
3,42,40716.81,Sur,Hombre
4,58,62881.04,Sur,Hombre


Dimensiones del DataFrame: (300, 4)


## Ejercicio 1: dispersión 3D

Se representa `edad` en el eje X, `ingresos` en el eje Y y `region` en el eje Z. El color distingue `genero`; el tema oscuro, el título centrado y el mayor tamaño de los marcadores completan la personalización solicitada.

In [ ]:
fig_3d = px.scatter_3d(
    df,
    x="edad",
    y="ingresos",
    z="region",
    color="genero",
    title="Edad, ingresos y región por género",
    template="plotly_dark",
)
fig_3d.update_traces(marker={"size": 6})
fig_3d.update_layout(title_x=0.5)
fig_3d.show()

## Ejercicio 2: boxplots de ingresos por región

Se añade un boxplot por región con `graph_objects`. La línea discontinua muestra la media global de ingresos, calculada sobre todas las filas. Los colores personalizados identifican cada región y la leyenda se titula `Regiones`.

In [ ]:
paleta_regiones = {
    "Norte": "#636EFA",
    "Centro": "#EF553B",
    "Sur": "#00CC96",
    "Este": "#AB63FA",
    "Oeste": "#FFA15A",
}
media_global = df["ingresos"].mean()
fig_box = go.Figure()

for region in regiones:
    ingresos_region = df.loc[df["region"] == region, "ingresos"]
    fig_box.add_trace(
        go.Box(
            y=ingresos_region,
            name=region,
            marker_color=paleta_regiones[region],
            legendgroup=region,
        )
    )

fig_box.add_hline(
    y=media_global,
    line_dash="dash",
    line_color="white",
    annotation_text=f"Media global: {media_global:,.0f}",
    annotation_position="top left",
)
fig_box.update_layout(
    title="Distribución de ingresos por región",
    title_x=0.5,
    xaxis_title="Región",
    yaxis_title="Ingresos",
    legend_title_text="Regiones",
    template="plotly_dark",
)
fig_box.show()

## Ejercicio 3: histograma y dispersión en subplots

Los dos gráficos comparten una leyenda de género: el histograma muestra la distribución de edades y el scatter relaciona edad e ingresos. Se muestra la leyenda una sola vez para evitar duplicados, mientras que los grupos y colores coinciden en ambos subplots.

In [ ]:
colores_genero = {"Mujer": "#FF6692", "Hombre": "#19D3F3"}
fig_subplots = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("Distribución de edades", "Edad e ingresos"),
    horizontal_spacing=0.12,
)

for genero in ["Mujer", "Hombre"]:
    grupo = df.loc[df["genero"] == genero]
    fig_subplots.add_trace(
        go.Histogram(
            x=grupo["edad"],
            name=genero,
            marker_color=colores_genero[genero],
            opacity=0.75,
            legendgroup=genero,
            showlegend=True,
        ),
        row=1,
        col=1,
    )
    fig_subplots.add_trace(
        go.Scatter(
            x=grupo["edad"],
            y=grupo["ingresos"],
            mode="markers",
            name=genero,
            marker={"color": colores_genero[genero], "size": 7, "opacity": 0.75},
            legendgroup=genero,
            showlegend=False,
        ),
        row=1,
        col=2,
    )

fig_subplots.update_layout(
    title="Comparación de edades e ingresos",
    title_x=0.5,
    barmode="overlay",
    legend_title_text="Género",
    template="plotly_dark",
)
fig_subplots.update_xaxes(title_text="Edad", row=1, col=1)
fig_subplots.update_xaxes(title_text="Edad", row=1, col=2)
fig_subplots.update_yaxes(title_text="Frecuencia", row=1, col=1)
fig_subplots.update_yaxes(title_text="Ingresos", row=1, col=2)
fig_subplots.show()

## Ejercicio 4: heatmap de densidad

`px.density_heatmap` agrupa las observaciones en intervalos de edad e ingresos. Las celdas con el color más intenso indican los intervalos donde se concentra un mayor número de personas. En los datos de demostración, observa el máximo del mapa para identificar esos rangos; con un dataset real, la interpretación debe basarse en su distribución, no en la tendencia usada para generar estos datos.

In [ ]:
fig_densidad = px.density_heatmap(
    df,
    x="edad",
    y="ingresos",
    nbinsx=18,
    nbinsy=18,
    color_continuous_scale="Viridis",
    title="Densidad conjunta de edad e ingresos",
)
fig_densidad.update_layout(
    title_x=0.5,
    xaxis_title="Edad",
    yaxis_title="Ingresos",
    coloraxis_colorbar_title="Personas",
)
fig_densidad.show()

## Ejercicios: interactividad y comparación

Estos ejercicios amplían el análisis con información al pasar el cursor, un control temporal, subplots comparables y facetas.

### Ejercicio 5: hover avanzado

El scatter muestra edad frente a ingresos. El hover incluye la diferencia firmada respecto a la media global: positiva si supera la media y negativa si queda por debajo.

In [ ]:
df["diferencia_media_ingresos"] = df["ingresos"] - df["ingresos"].mean()

fig_hover = px.scatter(
    df,
    x="edad",
    y="ingresos",
    color="genero",
    custom_data=["diferencia_media_ingresos"],
    title="Ingresos por edad: diferencia respecto a la media",
    labels={"edad": "Edad (años)", "ingresos": "Ingresos"},
)
fig_hover.update_traces(
    hovertemplate=(
        "<b>Edad:</b> %{x}<br>"
        "<b>Ingresos:</b> %{y:,.2f}<br>"
        "<b>Diferencia respecto a la media:</b> %{customdata[0]:+,.2f}"
        "<extra></extra>"
    )
)
fig_hover.update_layout(title_x=0.5)
fig_hover.show()

### Ejercicio 6: slider temporal de ventas

Se simulan 100 días de ventas para dos productos. El control de rango del eje X permite seleccionar el intervalo temporal que se quiere explorar.

In [ ]:
fechas_ventas = pd.date_range("2025-01-01", periods=100, freq="D")
dias = np.arange(len(fechas_ventas))
ventas_producto_a = (
    70 + 0.18 * dias + 9 * np.sin(2 * np.pi * dias / 7)
    + generador.normal(0, 5, size=len(dias))
).clip(0).round().astype(int)
ventas_producto_b = (
    52 + 0.12 * dias + 7 * np.sin(2 * np.pi * dias / 7 + 1)
    + generador.normal(0, 4, size=len(dias))
).clip(0).round().astype(int)

df_ventas = pd.DataFrame(
    {
        "fecha": fechas_ventas,
        "Producto A": ventas_producto_a,
        "Producto B": ventas_producto_b,
    }
)
fig_ventas = px.line(
    df_ventas,
    x="fecha",
    y=["Producto A", "Producto B"],
    title="Ventas diarias durante 100 días",
    labels={"fecha": "Fecha", "value": "Unidades vendidas", "variable": "Producto"},
)
fig_ventas.update_xaxes(
    rangeslider_visible=True,
    rangeselector={
        "buttons": [
            {"count": 14, "label": "14 días", "step": "day", "stepmode": "backward"},
            {"count": 30, "label": "30 días", "step": "day", "stepmode": "backward"},
            {"step": "all", "label": "Todo"},
        ]
    },
)
fig_ventas.update_layout(title_x=0.5)
fig_ventas.show()

### Ejercicio 7: histograma y boxplots con escala Y compartida

El histograma se dibuja horizontalmente para que su eje Y represente los ingresos, igual que en los boxplots por región. Así ambos subplots comparten una escala de ingresos comparable; el eje X representa frecuencia a la izquierda y región a la derecha.

In [ ]:
fig_comparativo = make_subplots(
    rows=1,
    cols=2,
    shared_yaxes=True,
    column_widths=[0.58, 0.42],
    subplot_titles=("Histograma de ingresos", "Ingresos por región"),
    horizontal_spacing=0.1,
)
fig_comparativo.add_trace(
    go.Histogram(
        y=df["ingresos"],
        orientation="h",
        nbinsy=20,
        marker_color="#636EFA",
        name="Frecuencia",
        showlegend=False,
    ),
    row=1,
    col=1,
)
for region in regiones:
    fig_comparativo.add_trace(
        go.Box(
            y=df.loc[df["region"] == region, "ingresos"],
            name=region,
            marker_color=paleta_regiones[region],
            legendgroup=region,
        ),
        row=1,
        col=2,
    )
fig_comparativo.update_xaxes(title_text="Número de personas", row=1, col=1)
fig_comparativo.update_xaxes(title_text="Región", row=1, col=2)
fig_comparativo.update_yaxes(title_text="Ingresos", row=1, col=1)
fig_comparativo.update_layout(
    title="Distribución y comparación de ingresos",
    title_x=0.5,
    legend_title_text="Regiones",
    template="plotly_dark",
)
fig_comparativo.show()

### Ejercicio 8: faceting por nivel de estudios

Se añade `nivel_estudios` con las categorías solicitadas y se crea un panel por nivel. El orden de las facetas va de Alto a Bajo de arriba abajo para facilitar la lectura.

In [ ]:
df["nivel_estudios"] = generador.choice(
    ["Bajo", "Medio", "Alto"],
    size=len(df),
    p=[0.25, 0.5, 0.25],
)
fig_facetas = px.scatter(
    df,
    x="edad",
    y="ingresos",
    facet_row="nivel_estudios",
    color="genero",
    category_orders={"nivel_estudios": ["Alto", "Medio", "Bajo"]},
    title="Edad e ingresos por nivel de estudios y género",
    labels={"edad": "Edad (años)", "ingresos": "Ingresos"},
)
fig_facetas.update_layout(title_x=0.5)
fig_facetas.show()

## Ejercicios: dashboards con Dash

Los tres siguientes ejercicios crean aplicaciones Dash y las muestran en el notebook. Ejecuta las celdas en orden; cada `app.run` inicia una vista interactiva.

### Ejercicio 1: países por continente

Se utiliza el dataset Gapminder de 2007 y se cuenta cada país una sola vez. El dashboard contiene un título y un gráfico de barras.

In [ ]:
from dash import Dash, dcc, html, Input, Output

df_gapminder_2007 = px.data.gapminder().query("year == 2007").copy()
paises_por_continente = (
    df_gapminder_2007.groupby("continent", as_index=False)
    .agg(numero_paises=("country", "nunique"))
    .sort_values("continent")
)

app_dash_basico = Dash(__name__)
fig_paises = px.bar(
    paises_por_continente,
    x="continent",
    y="numero_paises",
    labels={"continent": "Continente", "numero_paises": "Número de países"},
    title="Número de países por continente (2007)",
)
app_dash_basico.layout = html.Div(
    [
        html.H1("Países por continente", style={"textAlign": "center"}),
        dcc.Graph(figure=fig_paises),
    ],
    style={"maxWidth": "1000px", "margin": "0 auto"},
)
app_dash_basico.run(jupyter_mode="inline", debug=False)

### Ejercicio 2: dashboard personalizado

Se centra el título, se asigna una paleta personalizada por continente y se añade una breve explicación bajo el gráfico.

In [ ]:
paleta_continentes = {
    "Africa": "#E69F00",
    "Americas": "#56B4E9",
    "Asia": "#009E73",
    "Europe": "#CC79A7",
    "Oceania": "#D55E00",
}
fig_paises_personalizado = px.bar(
    paises_por_continente,
    x="continent",
    y="numero_paises",
    color="continent",
    color_discrete_map=paleta_continentes,
    labels={"continent": "Continente", "numero_paises": "Número de países"},
    title="Número de países por continente (2007)",
)
app_dash_personalizado = Dash(__name__)
app_dash_personalizado.layout = html.Div(
    [
        html.H1("Países por continente", style={"textAlign": "center"}),
        dcc.Graph(figure=fig_paises_personalizado),
        html.P(
            "El gráfico compara cuántos países de la muestra de Gapminder pertenecen a cada continente en 2007.",
            style={"textAlign": "center"},
        ),
    ],
    style={"maxWidth": "1000px", "margin": "0 auto"},
)
app_dash_personalizado.run(jupyter_mode="inline", debug=False)

### Ejercicio 3: dashboard interactivo con filtro

El menú desplegable filtra el scatter por continente. Cada punto representa un país; el tamaño indica la población y el hover incluye el nombre del país.

In [ ]:
continentes = sorted(df_gapminder_2007["continent"].unique())
app_dash_interactivo = Dash(__name__)
app_dash_interactivo.layout = html.Div(
    [
        html.H1("PIB y esperanza de vida", style={"textAlign": "center"}),
        dcc.Dropdown(
            id="selector-continente",
            options=[{"label": continente, "value": continente} for continente in continentes],
            value=continentes[0],
            clearable=False,
        ),
        dcc.Graph(id="grafico-pib-esperanza"),
    ],
    style={"maxWidth": "1000px", "margin": "0 auto"},
)

@app_dash_interactivo.callback(
    Output("grafico-pib-esperanza", "figure"),
    Input("selector-continente", "value"),
)
def actualizar_grafico(continente_seleccionado):
    datos = df_gapminder_2007.loc[
        df_gapminder_2007["continent"] == continente_seleccionado
    ]
    return px.scatter(
        datos,
        x="gdpPercap",
        y="lifeExp",
        size="pop",
        hover_name="country",
        labels={
            "gdpPercap": "PIB per cápita (USD por persona)",
            "lifeExp": "Esperanza de vida (años)",
        },
        title=f"{continente_seleccionado}: PIB y esperanza de vida (2007)",
    )

app_dash_interactivo.run(jupyter_mode="inline", debug=False)

## Ejercicios: selección de gráficos, paletas y escalas

### Ejercicio 1: elegir un gráfico para los errores y la precisión

- **Errores por clase:** usaría un gráfico de barras, porque las clases son categorías discretas y se comparan sus recuentos.
- **Precisión por clase:** también usaría barras para comparar directamente el valor de cada clase. Un gráfico de líneas podría sugerir una continuidad u orden cuantitativo entre las clases que aquí no existe.

In [1]:
import pandas as pd
import plotly.express as px

df_errors = pd.DataFrame(
    {
        "clase": list(range(10)),
        "errores": [50, 40, 35, 70, 20, 15, 30, 25, 45, 60],
        "precision": [0.95, 0.93, 0.90, 0.85, 0.97, 0.98, 0.92, 0.91, 0.89, 0.87],
    }
)
fig_errores_clase = px.bar(
    df_errors,
    x="clase",
    y="errores",
    title="Errores de clasificación por clase",
    labels={"clase": "Clase", "errores": "Número de errores"},
)
fig_errores_clase.update_layout(title_x=0.5)
fig_errores_clase.show()

fig_precision_clase = px.bar(
    df_errors,
    x="clase",
    y="precision",
    title="Precisión por clase",
    labels={"clase": "Clase", "precision": "Precisión"},
)
fig_precision_clase.update_yaxes(range=[0, 1])
fig_precision_clase.update_layout(title_x=0.5)
fig_precision_clase.show()

### Ejercicio 2: scatter de Iris con paleta segura para daltonismo

`px.colors.qualitative.Safe` es una paleta cualitativa diseñada para distinguir categorías con mayor accesibilidad cromática. Cada especie recibe un color.

In [ ]:
import plotly.express as px

df_iris = px.data.iris()
fig_iris = px.scatter(
    df_iris,
    x="sepal_width",
    y="sepal_length",
    color="species",
    color_discrete_sequence=px.colors.qualitative.Safe,
    title="Ancho y largo del sépalo por especie",
    labels={
        "sepal_width": "Ancho del sépalo (cm)",
        "sepal_length": "Largo del sépalo (cm)",
        "species": "Especie",
    },
)
fig_iris.update_layout(title_x=0.5)
fig_iris.show()

### Ejercicio 3: PIB per cápita y esperanza de vida en escala logarítmica

Se usa el dataset Gapminder de 2007. La escala logarítmica permite apreciar mejor países con niveles de PIB muy distintos; los títulos de ejes incluyen las unidades.

In [ ]:
import plotly.express as px

df_gapminder_log = px.data.gapminder().query("year == 2007")
fig_gapminder_log = px.scatter(
    df_gapminder_log,
    x="gdpPercap",
    y="lifeExp",
    color="continent",
    hover_name="country",
    log_x=True,
    title="PIB per cápita y esperanza de vida (2007)",
    labels={
        "gdpPercap": "PIB per cápita (USD por persona, escala logarítmica)",
        "lifeExp": "Esperanza de vida (años)",
        "continent": "Continente",
    },
)
fig_gapminder_log.update_layout(title_x=0.5)
fig_gapminder_log.show()